# Подбор гиперпараметров градиентного бустинга (CatBoost)
Подбор гиперпараметров модели градиентного бустинга по аналогии с `grad_boost.ipynb` и подбором параметров логистической регрессии из `log_reg.ipynb` (перебор через `ParameterGrid` + логирование в файл).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import logging
from datetime import datetime
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    balanced_accuracy_score, accuracy_score, precision_score,
    recall_score, f1_score, log_loss
)
from catboost import CatBoostClassifier

In [ ]:
DataFileName = "datasets/train.csv" 
df = pd.read_csv(DataFileName)
M, N = df.shape
categorical_features = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']

# сначала найдём все потенциально невалидные данные и выбросы
borders = {"sleep_duration": [0.0, 24.0], "heart_rate": [0.0, 220.0], "bmi": [0.0, 100],
           "calorie_expenditure": [0.0, np.inf], "step_count": [0.0, np.inf], 
           "exercise_duration": [0.0, 1440.0], "water_intake": [0.0, np.inf],
           "diet_type": ("veg", "non-veg", "balanced"), "stress_level": ("low", "high", "medium"),
           "sleep_quality": ('average', 'poor', 'good'), "physical_activity_level": ('sedentary', 'moderate', 'active'),
           "smoking_alcohol": ('yes', 'occasional', 'no'), "gender": ('female', 'other', 'male')}

for col_name in df.columns:
    if col_name not in borders:
        continue
    else:
        try:
            if type(borders[col_name]) is tuple:
                mask = ~(df[col_name].isin(borders[col_name]) | df[col_name].isna())
            elif type(borders[col_name]) is list:
                mask = ~(((df[col_name] >= borders[col_name][0]) & (df[col_name] <= borders[col_name][1])) | df[col_name].isna())
            else:
                raise AttributeError("Нестандартный тип данных из словаря borders")
        except AttributeError as ae:
            print(f"Словарь borders повреждён: {ae}")
        print(f"Столбец {col_name}: {mask.sum()}")

### Вывод
Данные лежат в валидных интервалах

In [ ]:
missing_per_row = df.isnull().sum(axis=1)
missing_counts = [[num, np.round(num/M, 2)] for num in missing_per_row.value_counts().sort_index()]
print("Количество строк с определенным числом пропусков (кол-во, процент от всего df):")
print(*missing_counts, sep="\n")

In [ ]:
# так как общий процент сэмплов с 3+ пропусками <= 0.2, то откинем их
df = df[~(df.isnull().sum(axis=1) >= 3)]

## Подготовка данных к обучению
Заполняем категориальные признаки, кодируем целевой признак и разбиваем на train/test (как в `grad_boost.ipynb`).

In [ ]:
df[categorical_features] = df[categorical_features].fillna("Unknown").astype(str)

le = LabelEncoder()
df["health_condition"] = le.fit_transform(df["health_condition"])

# Разделяем на train / test
y = df["health_condition"]
df_train, df_test = train_test_split(df, stratify=y, random_state=42, test_size=0.2)

X_train = df_train.drop(columns=["health_condition", "id"])
y_train = df_train["health_condition"] 

X_test = df_test.drop(columns=["health_condition", "id"])
y_test = df_test["health_condition"]

## Подбор гиперпараметров
Подбор ведём на всей обучающей выборке (с внутренней валидацией), перебором сетки параметров `ParameterGrid` с логированием результатов в `logs_grad_boost`. Основная метрика — `bal_acc` (balanced accuracy).

In [ ]:
# Подбор ведём на всей обучающей выборке df_train с внутренней валидацией
df_search_train, df_search_val = train_test_split(
    df_train, stratify=df_train["health_condition"], random_state=42, test_size=0.15
)

X_search_train = df_search_train.drop(columns=["health_condition", "id"])
y_search_train = df_search_train["health_condition"]
X_search_val = df_search_val.drop(columns=["health_condition", "id"])
y_search_val = df_search_val["health_condition"]

param_grid = {
    "learning_rate": [0.05, 0.1, 0.2],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [3, 5],
}

In [ ]:
log_dir = 'logs_grad_boost'
os.makedirs(log_dir, exist_ok=True)

log_filename = os.path.join(
    log_dir, 
    f'training_log_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename)
    ],
    force=True 
)

results = []
model_num = 0

for params in ParameterGrid(param_grid):
    logging.info("=" * 60)
    logging.info(f"Model_{model_num}; " + ";".join([f"{k}={v}" for k, v in params.items()]))

    model = CatBoostClassifier(
        iterations=200,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        early_stopping_rounds=50,
        random_seed=42,
        verbose=0,
        **params
    )

    model.fit(
        X_search_train, y_search_train,
        cat_features=categorical_features,
        eval_set=(X_search_val, y_search_val),
        verbose=0
    )

    y_pred = model.predict(X_search_val)

    cur_bal_acc = balanced_accuracy_score(y_search_val, y_pred)
    cur_acc = accuracy_score(y_search_val, y_pred)
    cur_f1 = f1_score(y_search_val, y_pred, average='macro', zero_division=0)
    cur_prec = precision_score(y_search_val, y_pred, average='macro', zero_division=0)
    cur_recall = recall_score(y_search_val, y_pred, average='macro', zero_division=0)
    best_iter = model.get_best_iteration()
    cur_loss = model.get_best_score()["validation"]["MultiClass"]

    results.append({
        **params,
        "best_iteration": best_iter,
        "log_loss": cur_loss,
        "bal_acc": cur_bal_acc,
        "accuracy": cur_acc,
        "f1_macro": cur_f1,
        "precision_macro": cur_prec,
        "recall_macro": cur_recall,
    })

    logging.info(f"best_iteration: {best_iter}")
    logging.info(f"log_loss: {cur_loss:.6f}")
    logging.info(f"bal_acc: {cur_bal_acc:.6f} | accuracy: {cur_acc:.6f} | f1_macro: {cur_f1:.6f}")
    logging.info(f"precision_macro: {cur_prec:.6f} | recall_macro: {cur_recall:.6f}")
    logging.info("=" * 60)

    model_num += 1

results_df = pd.DataFrame(results)
results_df.to_csv(
    os.path.join(log_dir, f'results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'),
    index=False
)

In [ ]:
print("Результаты подбора гиперпараметров (сортировка по bal_acc):")
print(results_df.sort_values("bal_acc", ascending=False).to_string(index=False))

best_row = results_df.sort_values("bal_acc", ascending=False).iloc[0]
best_hyperparams = {
    "learning_rate": float(best_row["learning_rate"]),
    "depth": int(best_row["depth"]),
    "l2_leaf_reg": int(best_row["l2_leaf_reg"]),
}
print("\nЛучшие гиперпараметры:", best_hyperparams)
print(f"Лучший bal_acc на валидации: {best_row['bal_acc']:.6f}")

In [ ]:
plt.figure(figsize=(12, 6))
sorted_df = results_df.sort_values("bal_acc", ascending=False).reset_index(drop=True)
plt.bar(range(len(sorted_df)), sorted_df["bal_acc"], color="#4ECDC4", alpha=0.8)
plt.xticks(
    range(len(sorted_df)),
    [f"lr={r['learning_rate']},d={r['depth']},l2={r['l2_leaf_reg']}" for _, r in sorted_df.iterrows()],
    rotation=45,
    ha="right",
    fontsize=8,
)
plt.ylabel("Balanced Accuracy")
plt.title("Balanced Accuracy по комбинациям гиперпараметров")
plt.tight_layout()
plt.show()

### Вывод
Лучшая комбинация гиперпараметров выбрана по `bal_acc` на валидации и выведена в предыдущей ячейке.

## Финальное обучение на полных данных с лучшими гиперпараметрами

In [ ]:
model = CatBoostClassifier(
    iterations=1000,
    loss_function='MultiClass',
    eval_metric='MultiClass',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100,
    **best_hyperparams
)

model.fit(
    X_train, y_train,
    cat_features=categorical_features,
    eval_set=(X_test, y_test),
    verbose=100
)

In [ ]:
if not os.path.exists("submission"):
    os.makedirs("submission")

df = pd.read_csv("datasets/test.csv")
y_id = df["id"]
df.drop(columns=["id"], inplace=True)
df[categorical_features] = df[categorical_features].fillna("Unknown").astype(str)

y_pred = le.inverse_transform(model.predict(df))
df_pred = pd.DataFrame({"health_condition": y_pred}, index=y_id)
df_pred.to_csv("submission/grad_bust_hyperparams.csv")